In [ ]:
import sys, re, os, json
from ChemCoTBench.baseline_and_eval.postprocess_eval.rxnutils import read_json, is_valid_smiles
from eval.utils import tranform_str_to_json

In [6]:
os.environ['http_proxy'] = 'http://127.0.0.1:7897'
os.environ['https_proxy'] = 'http://127.0.0.1:7897'
from evaluator import MoleculeSMILESEvaluator
evaluator = MoleculeSMILESEvaluator()

[nltk_data] Downloading package wordnet to /remote-
[nltk_data]     home/myc/nltk_data...


In [19]:
subtask_to_result_key = {
    "RCR": "SMILES",
    "nepp": "pred_smi",
    "mechsel": "choice",
    "major_product": "Major Product",
    "byproduct": "Byproduct(s)",
    "retro": "Reactants"
}

In [25]:
def evaluate_mol(model_name: str, subtask: str, log_dir: str = None):
    print(f'{model_name} {subtask}')
    if log_dir is None:
        log_dir = f"logs/{subtask}"
    
    if not os.path.exists(log_dir):
        raise ValueError(f"logs_dir {log_dir} is not correct")
    samples = read_json(f"{log_dir}/{model_name}.json")
    preds = []
    gts = []
    for sample in samples:
        gt = sample['gt']
        if subtask in ['major_product', 'byproduct']:
            gt = json.loads(gt)
            gts.append(gt[subtask_to_result_key[subtask]])
        elif subtask == 'retro':
            if len(gt) == 0:
                continue
            gts.append('.'.join(gt))
        else:
            gts.append(gt)

        try:
            pred_smiles = tranform_str_to_json(sample['json_response'])
            pred = pred_smiles[subtask_to_result_key[subtask]]
            if subtask == 'retro':
                pred = '.'.join(pred)
            preds.append(pred)
        except Exception as e:
            print(f'error parsing {sample['json_response']}: {e}')
            preds.append('')
        
    res = evaluator.evaluate(preds, gts)
    if subtask in ['RCR', 'major_product', 'byproduct', 'retro']:
        fts = (res['rdk_sims'] + res['maccs_sims'] + res['morgan_sims']) / 3
        res['fts'] = fts
        
    return res

In [10]:
def evaluate_MechSel(model_name: str, logs_dir: str = 'logs/mechsel'):
    """
    Evaluate the reaction mechanism selection prediction.

    Args:
        logs_dir (str): The directory where the logs are stored.
        model_name (str): The name of the model.

    Returns:
        None
    """
    if not os.path.exists(logs_dir):
        raise ValueError(f"logs_dir {logs_dir} is not correct")
    samples = read_json(f"{logs_dir}/{model_name}.json")

    preds = []
    gts = []
    for sample in samples:
        pred_smiles = tranform_str_to_json(sample['json_response'])
        pred_choice = pred_smiles[subtask_to_result_key['MechSel']]
        preds.append(pred_choice)
        if len(pred_choice) > 1:
            # if multiple chars, we take the first one
            pred_choice = pred_choice[0]
        # if pred_choice is not a valid choice, we treat it as empty
        if not pred_choice.lower().isalpha():
            pred_choice = ""

        pred_choice = pred_choice.lower()
        gt = sample['gt'].lower()
        preds.append(pred_choice)
        gts.append(gt)

    accuracy = sum(1 for pred, gt in zip(preds, gts) if pred == gt) / len(gts)
    return {"MCQ Accuracy (mean)": accuracy}

In [11]:
def evaluate_all_subtasks(model_name: str, logs_dir: str = 'logs'):
    all_results = {}
    subtasks = subtask_to_result_key.keys()
    for subtask in subtasks:
        if subtask == 'MechSel':
            all_results[subtask] = evaluate_MechSel(model_name)
        elif subtask in ['major_product', 'byproduct']:
            all_results[subtask] = evaluate_mol(model_name, subtask, f"{logs_dir}/fs")
        else:
            all_results[subtask] = evaluate_mol(model_name, subtask)
    print(f"eval_score_{model_name}", all_results)

In [26]:
if __name__ == '__main__':
    models = ['qwen3-8b']
    for model in models:
        evaluate_all_subtasks(model)

qwen3-8b RCR
error parsing ```json
{
    "SMILES": "CC(C)(C)OC(=O)N1C[C@H](C(=O)O)C[C@@H]1C(=O)N2C[C@H](C(=O)O)C[C@H]2C(=O)N3C[C@H](C(=O)O)C[C@H]3C(=O)N4C[C@H](C(=O)O)C[C@H]4C(=O)N5C[C@H](C(=O)O)C[C@H]5C(=O)N6C[C@H](C(=O)O)C[C@H]6C(=O)N7C[C@H](C(=O)O)C[C@H]7C(=O)N8C[C@H](C(=O)O)C[C@H]8C(=O)N9C[C@H](C(=O)O)C[C@H]9C(=O)N10C[C@H](C(=O)O)C[C@H]10C(=O)N11C[C@H](C(=O)O)C[C@H]11C(=O)N12C[C@H](C(=O)O)C[C@H]12C(=O)N13C[C@H](C(=O)O)C[C@H]13C(=O)N14C[C@H](C(=O)O)C[C@H]14C(=O)N15C[C@H](C(=O)O)C[C@H]15C(=O)N16C[C@H](C(=O)O)C[C@H]16C(=O)N17C[C@H](C(=O)O)C[C@H]17C(=O)N18C[C@H](C(=O)O)C[C@H]18C(=O)N19C[C@H](C(=O)O)C[C@H]19C(=O)N20C[C@H](C(=O)O)C[C@H]20C(=O)N21C[C@H](C(=O)O)C[C@H]21C(=O)N22C[C@H](C(=O)O)C[C@H]22C(=O)N23C[C@H](C(=O)O)C[C@H]23C(=O)N24C[C@H](C(=O)O)C[C@H]24C(=O)N25C[C@H](C(=O)O)C[C@H]25C(=O)N26C[C@H](C(=O)O)C[C@H]26C(=O)N27C[C@H](C(=O)O)C[C@H]27C(=O)N28C[C@H](C(=O)O)C[C@H]28C(=O)N29C[C@H](C(=O)O)C[C@H]29C(=O)N30C[C@H](C(=O)O)C[C@H]30C(=O)N31C[C@H](C(=O)O)C[C@H]31C(=O)N32C[C@H](C(=O)O)C[C@